In [ ]:
import cv2
import mediapipe as mp
import time
import random
import warnings
from ultralytics import YOLO
from collections import Counter

warnings.filterwarnings("ignore", category=UserWarning, module='google.protobuf')
model = YOLO("face_filter.pt")
mp_face = mp.solutions.face_mesh
face_mesh = mp_face.FaceMesh(static_image_mode=False, max_num_faces=1)

cap = cv2.VideoCapture(0)
emociones_filtro = ["tristeza", "miedo", "ira", "alegria"]
imagenes = {}

for emo in emociones_filtro:
    img = cv2.imread(f"imagenes/{emo}.jpg")
    if img is not None:
        img = cv2.resize(img, (150, 150))
        imagenes[emo] = img

mapeo_emociones = {
    "neutral": "neutral",
    "happy": "alegria",
    "sad": "tristeza", 
    "angry": "ira",
    "fear": "miedo",
    "disgust": "ira",
    "surprise": "alegria"
}

estado_giratorio = 0
estado_final = 1
estado_actual = estado_giratorio
inicio_tiempo = time.time()
tiempo_limite = 10  # 10 segundos girando las imagenes
emocion_filtro_actual = random.choice(emociones_filtro)
ultimo_cambio = time.time()
intervalo_cambio = 0.2

emociones_detectadas = []
ultimo_reconocimiento = time.time()
intervalo_reconocimiento = 1.0  # cada segundo capturamos los sentimientos
emocion_final = "..."
emocion_detectada_actual = "..."

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    resultados = face_mesh.process(rgb)
    h, w, _ = frame.shape
    if estado_actual == estado_giratorio:
        if time.time() - ultimo_cambio > intervalo_cambio:
            emocion_filtro_actual = random.choice(emociones_filtro)
            ultimo_cambio = time.time()
        if time.time() - inicio_tiempo > tiempo_limite:
            if emociones_detectadas:
                contador = Counter(emociones_detectadas)
                emocion_mas_frecuente = contador.most_common(1)[0][0]
                emocion_final = mapeo_emociones.get(emocion_mas_frecuente, "alegria")
            else:
                emocion_final = random.choice(emociones_filtro)
            
            estado_actual = estado_final
            emocion_filtro_actual = emocion_final
    
    if resultados.multi_face_landmarks:
        for rostro in resultados.multi_face_landmarks:
            xs = [int(l.x * w) for l in rostro.landmark]
            ys = [int(l.y * h) for l in rostro.landmark]

            x_min, x_max = max(min(xs), 0), min(max(xs), w)
            y_min, y_max = max(min(ys), 0), min(max(ys), h)

            # Recortar el rostro
            cara = frame[y_min:y_max, x_min:x_max]

            # Ejecutamos el modelo YOLO cada segundo 
            if estado_actual == estado_giratorio and time.time() - ultimo_reconocimiento > intervalo_reconocimiento and cara.size > 0:
                results = model.predict(cara, conf=0.1, verbose=False)
                if results and len(results[0].boxes) > 0:
                    labels = results[0].boxes.cls.cpu().numpy().astype(int)
                    names = results[0].names
                    emocion = names[labels[0]]
                    emocion_detectada_actual = mapeo_emociones.get(emocion, emocion)
                    emociones_detectadas.append(emocion)
                ultimo_reconocimiento = time.time()

            # Mostrar filtro encima de la cabeza
            punto_cabeza = rostro.landmark[10]
            x = int(punto_cabeza.x * w)
            y = int(punto_cabeza.y * h)

            img_emocion = imagenes.get(emocion_filtro_actual)
            if img_emocion is not None:
                img_h, img_w, _ = img_emocion.shape
                y1 = max(0, y - img_h - 100)
                y2 = min(h, y1 + img_h)
                x1 = max(0, x - img_w // 2)
                x2 = min(w, x1 + img_w)
                if y2 > y1 and x2 > x1:
                    roi = frame[y1:y2, x1:x2]
                    if roi.shape[:2] == img_emocion.shape[:2]:
                        img_gray = cv2.cvtColor(img_emocion, cv2.COLOR_BGR2GRAY)
                        _, mask = cv2.threshold(img_gray, 10, 255, cv2.THRESH_BINARY)
                        mask_inv = cv2.bitwise_not(mask)
                        fondo = cv2.bitwise_and(roi, roi, mask=mask_inv)
                        frente = cv2.bitwise_and(img_emocion, img_emocion, mask=mask)
                        frame[y1:y2, x1:x2] = cv2.add(fondo, frente)


    if estado_actual == estado_giratorio:
        tiempo_restante = max(0, tiempo_limite - (time.time() - inicio_tiempo))
        texto = f"Analizando sentimientos... {int(tiempo_restante)}s"
    else:
        texto = "Sentimiento detectado!"

    cv2.putText(frame, texto, (40, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)
    cv2.putText(frame, texto, (40, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 2)

    if estado_actual == estado_giratorio and emocion_detectada_actual != "...":
        cv2.rectangle(frame, (40, h - 100), (400, h - 40), (0, 0, 0), -1)
        cv2.putText(frame, f"Detectado: {emocion_detectada_actual.upper()}",
                    (60, h - 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                    (255, 255, 255), 2)
    elif estado_actual == estado_final:
        cv2.rectangle(frame, (40, h - 100), (400, h - 40), (0, 100, 0), -1)
        cv2.putText(frame, f"Tu sentimiento: {emocion_final.upper()}",
                    (60, h - 60), cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                    (255, 255, 255), 2)
    cv2.imshow("Filtro de sentimientos", frame)
    if cv2.waitKey(5) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import mediapipe as mp
import time
import random
import warnings
import numpy as np
from ultralytics import YOLO
from collections import Counter

warnings.filterwarnings("ignore", category=UserWarning, module='google.protobuf')

# Cargar modelo YOLO
model = YOLO("arma_definitiva_definitiva.pt")

# Inicializar MediaPipe FaceMesh
mp_face = mp.solutions.face_mesh
face_mesh = mp_face.FaceMesh(static_image_mode=False, max_num_faces=1)

# Inicializar MediaPipe Selfie Segmentation
mp_selfie = mp.solutions.selfie_segmentation
selfie_segmenter = mp_selfie.SelfieSegmentation(model_selection=1)

# Inicializar cámara
cap = cv2.VideoCapture(0)

# Cargar imágenes de emociones (para el filtro)
emociones_filtro = ["tristeza", "miedo", "ira", "alegria"]
imagenes = {}
for emo in emociones_filtro:
    img = cv2.imread(f"imagenes/{emo}.jpg")
    if img is not None:
        img = cv2.resize(img, (200, 150))
        imagenes[emo] = img

mapeo_emociones = {
    "neutral": "neutral",
    "happy": "alegria",
    "sad": "tristeza", 
    "angry": "ira",
    "fear": "miedo",
    "disgust": "ira", 
    "surprise": "alegria"
}

colores_emociones = {
    "ira": (0, 0, 255),        # Rojo
    "tristeza": (255, 0, 0),   # Azul
    "alegria": (0, 255, 255),  # Amarillo
    "miedo": (255, 0, 255),    # Lila/Magenta
    "disgust": (0, 128, 0),    # Verde
    "surprise": (255, 0, 255), # Lila/Magenta
    "neutral": (128, 128, 128) # Gris
}

estado_giratorio = 0
estado_final = 1

estado_actual = estado_giratorio
inicio_tiempo = time.time()
tiempo_limite = 10  # 10 segundos en modo giratorio
emocion_filtro_actual = random.choice(emociones_filtro)
ultimo_cambio = time.time()
intervalo_cambio = 0.2

# Para almacenar emociones detectadas
emociones_detectadas = []
ultimo_reconocimiento = time.time()
intervalo_reconocimiento = 1.0  # cada segundo
emocion_final = "..."
emocion_detectada_actual = "..."

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    resultados_segmentacion = selfie_segmenter.process(rgb)
    
    resultados = face_mesh.process(rgb)
    h, w, _ = frame.shape

    if estado_actual == estado_giratorio:
        if time.time() - ultimo_cambio > intervalo_cambio:
            emocion_filtro_actual = random.choice(emociones_filtro)
            ultimo_cambio = time.time()
        # Verificar si ha pasado el tiempo límite
        if time.time() - inicio_tiempo > tiempo_limite:
            if emociones_detectadas:
                contador = Counter(emociones_detectadas)
                emocion_mas_frecuente = contador.most_common(1)[0][0]
                emocion_final = mapeo_emociones.get(emocion_mas_frecuente, "alegria")
            else:
                emocion_final = random.choice(emociones_filtro)
            
            estado_actual = estado_final
            emocion_filtro_actual = emocion_final
    
    # Aplicar coloracion
    if estado_actual == estado_final and resultados_segmentacion.segmentation_mask is not None:
        mask = resultados_segmentacion.segmentation_mask > 0.5
        color_cuerpo = colores_emociones.get(emocion_final, (128, 128, 128))
        
        # Crear una imagen del color sólido
        colored_area = np.zeros_like(frame)
        colored_area[:] = color_cuerpo
        frame[mask] = cv2.addWeighted(frame[mask], 0.7, colored_area[mask], 0.7, 0)

    # Detectar rostro
    if resultados.multi_face_landmarks:
        for rostro in resultados.multi_face_landmarks:
            # Coordenadas del rostro completo
            xs = [int(l.x * w) for l in rostro.landmark]
            ys = [int(l.y * h) for l in rostro.landmark]

            x_min, x_max = max(min(xs), 0), min(max(xs), w)
            y_min, y_max = max(min(ys), 0), min(max(ys), h)
            cara = frame[y_min:y_max, x_min:x_max]

            # Ejecutar modelo 
            if estado_actual == estado_giratorio and time.time() - ultimo_reconocimiento > intervalo_reconocimiento and cara.size > 0:
                results = model.predict(cara, conf=0.1, verbose=False)
                if results and len(results[0].boxes) > 0:
                    labels = results[0].boxes.cls.cpu().numpy().astype(int)
                    names = results[0].names
                    emocion = names[labels[0]]
                    emocion_detectada_actual = mapeo_emociones.get(emocion, emocion)
                    emociones_detectadas.append(emocion)
                ultimo_reconocimiento = time.time()

            punto_cabeza = rostro.landmark[10]
            x = int(punto_cabeza.x * w)
            y = int(punto_cabeza.y * h)

            img_emocion = imagenes.get(emocion_filtro_actual)
            if img_emocion is not None:
                img_h, img_w, _ = img_emocion.shape
                y1 = max(0, y - img_h - 100)
                y2 = min(h, y1 + img_h)
                x1 = max(0, x - img_w // 2)
                x2 = min(w, x1 + img_w)
                if y2 > y1 and x2 > x1:
                    roi = frame[y1:y2, x1:x2]
                    if roi.shape[:2] == img_emocion.shape[:2]:
                        img_gray = cv2.cvtColor(img_emocion, cv2.COLOR_BGR2GRAY)
                        _, mask = cv2.threshold(img_gray, 10, 255, cv2.THRESH_BINARY)
                        mask_inv = cv2.bitwise_not(mask)
                        fondo = cv2.bitwise_and(roi, roi, mask=mask_inv)
                        frente = cv2.bitwise_and(img_emocion, img_emocion, mask=mask)
                        frame[y1:y2, x1:x2] = cv2.add(fondo, frente)

    if estado_actual == estado_giratorio:
        tiempo_restante = max(0, tiempo_limite - (time.time() - inicio_tiempo))
        texto = f"Analizando sentimientos... {int(tiempo_restante)}s"
    else:
        texto = ""

    cv2.putText(frame, texto, (40, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)
    cv2.putText(frame, texto, (40, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 2)

    if estado_actual == estado_giratorio and emocion_detectada_actual != "...":
        cv2.rectangle(frame, (40, h - 100), (400, h - 40), (0, 0, 0), -1)
        cv2.putText(frame, f"Detectado: {emocion_detectada_actual.upper()}",
                    (60, h - 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                    (255, 255, 255), 2)
    elif estado_actual == estado_final:
        cv2.rectangle(frame, (40, h - 100), (500, h - 40), (0, 0, 0), -1)
        cv2.putText(frame, f"Tu sentimiento: {emocion_final.upper()}",
                    (60, h - 60), cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                    (255, 255, 255), 2)
        
        color_actual = colores_emociones.get(emocion_final, (128, 128, 128))

    cv2.imshow("Filtro de sentimientos", frame)

    # Salir con ESC
    if cv2.waitKey(5) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()